In [0]:
%pip install kagglehub transformers torch torchvision pillow scikit-learn optuna mlflow timm --quiet
dbutils.library.restartPython()
# First Change

In [0]:
import kagglehub
import os
import shutil
from datetime import datetime

# Descargar dataset desde Kaggle
print("Descargando dataset AI vs Real Images desde Kaggle...")
path = kagglehub.dataset_download("rhythmghai/ai-vs-real-images-dataset")
print(f"Dataset descargado en: {path}")

# Definir ruta de capa bronze
bronze_path = "/tmp/medallion/bronze/ai_vs_real_images"
os.makedirs(bronze_path, exist_ok=True)

# Copiar datos a capa bronze
print(f"Copiando datos a capa bronze: {bronze_path}")
if os.path.exists(path):
    for item in os.listdir(path):
        src = os.path.join(path, item)
        dst = os.path.join(bronze_path, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

print(f"Datos copiados a capa bronze exitosamente!")
print(f"Contenido de capa bronze: {os.listdir(bronze_path)}")

# Mostrar estructura
for root, dirs, files in os.walk(bronze_path):
    level = root.replace(bronze_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            print(f'{subindent}{file}')
        if len(files) > 3:
            print(f'{subindent}... y {len(files)-3} archivos más')

In [0]:
from pyspark.sql.functions import col, lit
import os

# Escanear estructura de directorio bronze
bronze_base = "/tmp/medallion/bronze/ai_vs_real_images"
image_data = []

for root, dirs, files in os.walk(bronze_base):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, file)
            # Determinar etiqueta desde estructura de directorio
            # FIX: Usar nombres específicos de carpeta
            if 'real_dataset' in root.lower():
                label = 'REAL'
            elif 'ai_generated_dataset' in root.lower():
                label = 'FAKE'
            else:
                label = 'UNKNOWN'
            
            image_data.append({
                'file_path': full_path,
                'file_name': file,
                'label': label,
                'ingestion_timestamp': datetime.now()
            })

print(f"Encontradas {len(image_data)} imágenes")

# Crear DataFrame
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("label", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False)
])

bronze_df = spark.createDataFrame(image_data, schema=schema)

# Escribir a tabla Delta Bronze en el schema workspace
bronze_table = "workspace.bronze.ai_images_raw"
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"✓ Tabla Bronze '{bronze_table}' creada con {bronze_df.count()} registros")
print(f"\nDistribución de etiquetas en Bronze:")
display(bronze_df.groupBy("label").count().orderBy("label"))

In [0]:
from pyspark.sql.functions import col, when

# Leer desde bronze
bronze_df = spark.table("workspace.bronze.ai_images_raw")

# Crear codificación de etiquetas (REAL=0, FAKE=1)
# FIX: Sin validación UDF que falla en contexto distribuido
silver_df = bronze_df.withColumn(
    "label_encoded",
    when(col("label") == "REAL", 0)
    .when(col("label") == "FAKE", 1)
    .otherwise(-1)
).filter(col("label_encoded") != -1)  # Remover etiquetas desconocidas

# Escribir a tabla Delta Silver
silver_table = "workspace.silver.ai_images_clean"
silver_df.select(
    "file_path",
    "file_name",
    "label",
    "label_encoded",
    "ingestion_timestamp"
).write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"✓ Tabla Silver '{silver_table}' creada con {silver_df.count()} registros válidos")
print(f"\nDistribución de etiquetas en Silver:")
display(spark.table(silver_table).groupBy("label", "label_encoded").count().orderBy("label"))

In [0]:
from pyspark.sql.functions import rand, monotonically_increasing_id, when

# Leer desde silver
silver_df = spark.table("workspace.silver.ai_images_clean")

# Agregar columna de división aleatoria (80% train, 20% val)
gold_df = silver_df.withColumn("random", rand(seed=42))
gold_df = gold_df.withColumn(
    "split",
    when(col("random") < 0.8, "train").otherwise("val")
)

# Agregar ID de fila para seguimiento
gold_df = gold_df.withColumn("image_id", monotonically_increasing_id())

# Escribir a tabla Delta Gold
gold_table = "workspace.gold.ai_images_training"
gold_df.select(
    "image_id",
    "file_path",
    "file_name",
    "label",
    "label_encoded",
    "split",
    "ingestion_timestamp"
).write.format("delta").mode("overwrite").saveAsTable(gold_table)

print(f"✓ Tabla Gold '{gold_table}' creada con {gold_df.count()} registros")
print(f"\nDistribución de división en Gold:")
display(spark.table(gold_table).groupBy("split", "label").count().orderBy("split", "label"))

In [0]:
# Verificar todas las tablas medallion
print("=" * 60)
print("RESUMEN DE ARQUITECTURA MEDALLION")
print("=" * 60)

print("\n📊 CAPA BRONZE (Datos Crudos):")
print(f"   Tabla: workspace.bronze.ai_images_raw")
bronze_count = spark.table("workspace.bronze.ai_images_raw").count()
print(f"   Total de registros: {bronze_count:,}")
bronze_labels = spark.table("workspace.bronze.ai_images_raw").groupBy("label").count().collect()
for row in bronze_labels:
    print(f"   - {row['label']}: {row['count']:,}")

print("\n✨ CAPA SILVER (Datos Limpios):")
print(f"   Tabla: workspace.silver.ai_images_clean")
silver_count = spark.table("workspace.silver.ai_images_clean").count()
print(f"   Total de registros: {silver_count:,}")
silver_labels = spark.table("workspace.silver.ai_images_clean").groupBy("label", "label_encoded").count().collect()
for row in silver_labels:
    print(f"   - {row['label']} (encoded={row['label_encoded']}): {row['count']:,}")

print("\n🏆 CAPA GOLD (Listo para Training):")
print(f"   Tabla: workspace.gold.ai_images_training")
gold_count = spark.table("workspace.gold.ai_images_training").count()
print(f"   Total de registros: {gold_count:,}")
gold_splits = spark.table("workspace.gold.ai_images_training").groupBy("split", "label").count().collect()
for row in gold_splits:
    print(f"   - {row['split']}/{row['label']}: {row['count']:,}")

print("\n" + "=" * 60)
print("✓ Arquitectura Medallion completada exitosamente!")
print("=" * 60)

In [0]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# Load gold table into pandas for PyTorch training
print("Loading gold training data...")
gold_pd = spark.table("workspace.gold.ai_images_training").toPandas()

print(f"\n✓ Total images loaded: {len(gold_pd):,}")
print(f"  Train images: {len(gold_pd[gold_pd['split'] == 'train']):,}")
print(f"  Validation images: {len(gold_pd[gold_pd['split'] == 'val']):,}")
print(f"\n✓ Label distribution:")
print(gold_pd.groupby(['split', 'label']).size())

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")

In [0]:
# Custom Dataset class for AI vs Real images
class AIImageDataset(Dataset):
    """PyTorch Dataset for loading images from file paths with transforms"""
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'file_path']
        label = int(self.df.loc[idx, 'label_encoded'])  # Convert to int to ensure Long type
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Define transforms following Kaggle competition best practices
# ImageNet normalization for DINOv2 pretrained model
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = AIImageDataset(
    gold_pd[gold_pd['split'] == 'train'],
    transform=train_transform
)

val_dataset = AIImageDataset(
    gold_pd[gold_pd['split'] == 'val'],
    transform=val_transform
)

print(f"✓ Datasets created:")
print(f"  Train dataset: {len(train_dataset):,} images")
print(f"  Validation dataset: {len(val_dataset):,} images")

# Test loading one sample
test_img, test_label = train_dataset[0]
print(f"\n✓ Sample image shape: {test_img.shape}")
print(f"  Sample label: {test_label} ({'FAKE' if test_label == 1 else 'REAL'})")
print(f"  Label type: {type(test_label)}")


In [0]:
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel

# Load DINOv2 base model (following Kaggle competition approach)
model_name = "facebook/dinov2-base"
print(f"Loading pretrained {model_name}...")

processor = AutoImageProcessor.from_pretrained(model_name)
dinov2_backbone = AutoModel.from_pretrained(model_name)

# Freeze backbone layers (we'll only train the classification head)
for param in dinov2_backbone.parameters():
    param.requires_grad = False

print(f"✓ DINOv2 backbone loaded and frozen")

# Define classification head following competition architecture
class DINOv2Classifier(nn.Module):
    """DINOv2 with custom classification head for binary classification"""
    def __init__(self, backbone, num_classes=2, dropout=0.3):
        super(DINOv2Classifier, self).__init__()
        self.backbone = backbone
        
        # Classification head with dropout for regularization
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),  # DINOv2-base outputs 768 features
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, pixel_values):
        # Get backbone features (use CLS token)
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]  # Extract CLS token
        
        # Classification
        logits = self.classifier(cls_token)
        return logits

# Initialize model
model = DINOv2Classifier(dinov2_backbone, num_classes=2, dropout=0.3)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters: {total_params - trainable_params:,}")

In [0]:
import mlflow
import mlflow.pytorch
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve
from tqdm import tqdm

# Set MLflow experiment
mlflow.set_experiment("/Users/yltabord@gmail.com/AI-vs-Real-DINOv2-Competition")

# Training hyperparameters (optimized from Kaggle competition)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 15
WEIGHT_DECAY = 1e-5

print("Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Device: {device}")

# Create data loaders (num_workers=0 to avoid shared memory issues)
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0,
    pin_memory=False
)

print(f"\n✓ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"\n✓ Training setup complete!")

In [0]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(loader, desc="Training")
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(loader, desc="Validation")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Get probabilities for AUC calculation
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probability of class 1 (FAKE)
            preds = torch.argmax(outputs, dim=1)
            
            running_loss += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_auc = roc_auc_score(all_labels, all_probs)
    
    return epoch_loss, epoch_acc, epoch_auc, all_labels, all_preds, all_probs

print("✓ Training functions defined")

In [0]:
# Start MLflow run
with mlflow.start_run(run_name="DINOv2-Competition-Approach") as run:
    # Log hyperparameters
    mlflow.log_param("model_architecture", model_name)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("num_epochs", NUM_EPOCHS)
    mlflow.log_param("weight_decay", WEIGHT_DECAY)
    mlflow.log_param("optimizer", "AdamW")
    mlflow.log_param("scheduler", "CosineAnnealingLR")
    mlflow.log_param("dropout", 0.3)
    mlflow.log_param("train_size", len(train_dataset))
    mlflow.log_param("val_size", len(val_dataset))
    
    best_auc = 0.0
    best_epoch = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    
    print("\n" + "="*70)
    print("STARTING TRAINING - Target: 99.23% AUC (Kaggle Competition Winner)")
    print("="*70)
    
    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"{'='*70}")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, val_auc, val_labels, val_preds, val_probs = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update learning rate
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        # Log metrics to MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)
        mlflow.log_metric("val_auc", val_auc, step=epoch)
        mlflow.log_metric("learning_rate", current_lr, step=epoch)
        
        # Print epoch summary
        print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
        print(f"Learning Rate: {current_lr:.6f}")
        
        # Save best model
        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), '/tmp/best_dinov2_model.pth')
            print(f"\n✓ New best model saved! AUC: {val_auc:.4f} ({val_auc*100:.2f}%)")
    
    # Log final metrics
    mlflow.log_metric("best_val_auc", best_auc)
    mlflow.log_metric("best_epoch", best_epoch)
    
    # Load best model for final evaluation
    model.load_state_dict(torch.load('/tmp/best_dinov2_model.pth'))
    
    # Final validation with best model
    final_loss, final_acc, final_auc, final_labels, final_preds, final_probs = validate_epoch(
        model, val_loader, criterion, device
    )
    
    # Print final results
    print(f"\n{'='*70}")
    print(f"FINAL RESULTS - Best Model from Epoch {best_epoch}")
    print(f"{'='*70}")
    print(f"\n🎯 Target AUC (Kaggle Competition): 99.23%")
    print(f"🏆 Achieved AUC: {final_auc*100:.2f}%")
    print(f"📊 Accuracy: {final_acc*100:.2f}%")
    print(f"📈 Gap from target: {abs(99.23 - final_auc*100):.2f}%")
    
    if final_auc >= 0.99:
        print(f"\n✅ SUCCESS! Achieved 99%+ AUC!")
    else:
        print(f"\n⚠️  Note: Below 99% - Consider fine-tuning DINOv2 backbone layers")
    
    print(f"\n{'-'*70}")
    print("Classification Report:")
    print(f"{'-'*70}")
    print(classification_report(final_labels, final_preds, target_names=['REAL', 'FAKE'], digits=4))
    
    print(f"\n{'-'*70}")
    print("Confusion Matrix:")
    print(f"{'-'*70}")
    cm = confusion_matrix(final_labels, final_preds)
    print(f"              Predicted REAL  Predicted FAKE")
    print(f"Actual REAL      {cm[0][0]:6d}          {cm[0][1]:6d}")
    print(f"Actual FAKE      {cm[1][0]:6d}          {cm[1][1]:6d}")
    
    # Log model to MLflow
    mlflow.pytorch.log_model(model, "dinov2_classifier")
    
    print(f"\n{'='*70}")
    print(f"✓ Training complete! MLflow run ID: {run.info.run_id}")
    print(f"{'='*70}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Training History - Loss
axes[0, 0].plot(range(1, NUM_EPOCHS+1), history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(range(1, NUM_EPOCHS+1), history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(alpha=0.3)

# 2. Training History - Accuracy & AUC
ax2 = axes[0, 1]
ax2.plot(range(1, NUM_EPOCHS+1), history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(range(1, NUM_EPOCHS+1), history['val_acc'], 'r-', label='Val Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training & Validation Metrics', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', fontsize=11)
ax2.grid(alpha=0.3)

# Add AUC on secondary y-axis
ax2_twin = ax2.twinx()
ax2_twin.plot(range(1, NUM_EPOCHS+1), history['val_auc'], 'g--', label='Val AUC', linewidth=2)
ax2_twin.set_ylabel('AUC', fontsize=12, color='g')
ax2_twin.tick_params(axis='y', labelcolor='g')
ax2_twin.legend(loc='center right', fontsize=11)

# 3. ROC Curve
fpr, tpr, thresholds = roc_curve(final_labels, final_probs)
roc_auc = final_auc

axes[1, 0].plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1, 0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[1, 0].set_xlim([0.0, 1.0])
axes[1, 0].set_ylim([0.0, 1.05])
axes[1, 0].set_xlabel('False Positive Rate', fontsize=12)
axes[1, 0].set_ylabel('True Positive Rate', fontsize=12)
axes[1, 0].set_title(f'ROC Curve - AUC: {roc_auc*100:.2f}%', fontsize=14, fontweight='bold')
axes[1, 0].legend(loc="lower right", fontsize=11)
axes[1, 0].grid(alpha=0.3)

# Add competition target line
axes[1, 0].axhline(y=0.9923, color='red', linestyle=':', linewidth=2, label='Competition Target (99.23%)')

# 4. Confusion Matrix Heatmap
cm = confusion_matrix(final_labels, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1], 
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'],
            cbar_kws={'label': 'Count'}, annot_kws={'size': 14, 'weight': 'bold'})
axes[1, 1].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('True Label', fontsize=12)
axes[1, 1].set_xlabel('Predicted Label', fontsize=12)

# Add accuracy text
accuracy_text = f"Accuracy: {final_acc*100:.2f}%\nAUC: {final_auc*100:.2f}%"
axes[1, 1].text(0.5, -0.15, accuracy_text, ha='center', va='center', 
               transform=axes[1, 1].transAxes, fontsize=11, 
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('/tmp/dinov2_training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualizations created and saved!")
print(f"\n{'='*70}")
print("COMPETITION COMPARISON")
print(f"{'='*70}")
print(f"🏆 Kaggle Competition Winner: 99.23% AUC")
print(f"📊 Our Implementation: {final_auc*100:.2f}% AUC")
print(f"📈 Difference: {abs(99.23 - final_auc*100):.2f} percentage points")
print(f"{'='*70}")